### Agenda

- Read CSV file with headers and schema
- Extract columns and display data
- Filter datasets
- Rename and cast columns
- Select specific columns
- Order by column values
- Use aggregations: count, min, max, sum, avg
- Use distinct
- Answer simple business questions from the dataset

### Dataset
- San Francisco Fire Department Public Dataset

### Analysis Questions
- What were all the different types of fire calls in 2018?
- What months in 2018 had the highest number of fire calls?
- Which neighborhood in San Francisco generated the most fire calls in 2018?
- Which neighborhoods had the worst response times to fire calls in 2018?
- Which week in 2018 had the most fire calls?
- Is there a relationship between neighborhood, zip code, and number of fire calls?

In [0]:
from pyspark.sql import functions as F

In [0]:
file_path = "/Volumes/jenny-demo/my_schema/external_data_source_files/sf-fire-calls.csv"

### Read CSV file
First, read the CSV using the header row and let Spark infer the schema for quick exploration.

In [0]:
df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(file_path)
)

df.printSchema()

In [0]:
display(df)

### Explore the dataset
Check columns, number of columns, and sample records.

In [0]:
df.columns

In [0]:
len(df.columns)

In [0]:
type(df.columns)

In [0]:
df.show(10, False)

### Select and filter columns
Look at a smaller subset of the dataset and filter out medical incidents.

In [0]:
few_df = (
    df.select("IncidentNumber", "AvailableDtTm", "CallType")
      .where(F.col("CallType") != "Medical Incident")
)

display(few_df)

### Projections and filters
Same idea, written a little more cleanly.

In [0]:
few_fire_df = (
    df.select("IncidentNumber", "AvailableDtTm", "CallType")
      .where(F.col("CallType") != "Medical Incident")
)

display(few_fire_df)

In [0]:
few_fire_df = (
    df.select(["IncidentNumber", "AvailableDtTm", "CallType"])
      .where(F.col("CallType") != "Medical Incident")
)

display(few_fire_df)

### Rename and cast columns
Create a cleaner delay column and convert it to a numeric type.

In [0]:
display(df.select("Delay"))

In [0]:
renamed_df = (
    df.withColumn("ResponseDelayedinMins", F.col("Delay").cast("decimal(10,2)"))
)

display(
    renamed_df.where(F.col("ResponseDelayedinMins") > 5)
)

### Cast date/time columns
Convert string date columns into proper timestamp/date columns for analysis.

In [0]:
fire_ts_df = (
    renamed_df
    .withColumn("IncidentDate", F.to_timestamp(F.col("CallDate"), "MM/dd/yyyy"))
    .withColumn("OnWatchDate", F.to_timestamp(F.col("WatchDate"), "MM/dd/yyyy"))
    .withColumn("AvailableDtTS", F.to_timestamp(F.col("AvailableDtTm"), "MM/dd/yyyy hh:mm:ss a"))
    .drop("CallDate", "WatchDate", "AvailableDtTm")
)

display(fire_ts_df.select("IncidentDate", "OnWatchDate", "AvailableDtTS"))

### Check distinct years in the dataset

In [0]:
display(
    fire_ts_df
    .select(F.year("IncidentDate").alias("Year"))
    .distinct()
    .orderBy("Year")
)

### Aggregations
Count fire calls by call type.

In [0]:
display(
    fire_ts_df
    .select("CallType")
    .where(F.col("CallType").isNotNull())
    .groupBy("CallType")
    .count()
    .orderBy(F.col("count").desc())
)

### Aggregate metrics
Calculate total alarms and delay statistics.

In [0]:
display(
    fire_ts_df.select(
        F.sum("NumAlarms").alias("TotalNumAlarms"),
        F.avg("ResponseDelayedinMins").alias("AvgResponseDelay"),
        F.min("ResponseDelayedinMins").alias("MinResponseDelay"),
        F.max("ResponseDelayedinMins").alias("MaxResponseDelay")
    )
)

In [0]:
display(
    fire_ts_df
    .groupBy("CallType")
    .agg(
        F.avg("ResponseDelayedinMins").alias("AvgResponseDelay"),
        F.min("ResponseDelayedinMins").alias("MinResponseDelay"),
        F.max("ResponseDelayedinMins").alias("MaxResponseDelay")
    )
)

### Filter to 2018 only

In [0]:
fire_ts_df_2018 = fire_ts_df.where(F.year("IncidentDate") == 2018)

display(fire_ts_df_2018)

### What were all the different types of fire calls in 2018?

In [0]:
display(
    fire_ts_df_2018
    .select("CallType")
    .where(F.col("CallType").isNotNull())
    .distinct()
    .orderBy("CallType")
)

### What months in 2018 had the highest number of fire calls?

In [0]:
display(
    fire_ts_df_2018
    .select(F.month("IncidentDate").alias("IncidentMonth"), "CallType")
    .groupBy("IncidentMonth")
    .agg(F.count("CallType").alias("CountOfIncidents"))
    .orderBy(F.col("CountOfIncidents").desc(), F.col("IncidentMonth"))
)

### Which neighborhood in San Francisco generated the most fire calls in 2018?

In [0]:
display(
    fire_ts_df_2018
    .select("CallType", "Neighborhood")
    .where(F.col("Neighborhood").isNotNull())
    .groupBy("Neighborhood")
    .agg(F.count("CallType").alias("CountOfIncidents"))
    .orderBy(F.col("CountOfIncidents").desc())
)

### Which neighborhoods had the worst response times to fire calls in 2018?

In [0]:
display(
    fire_ts_df_2018
    .select("CallType", "Neighborhood", "ResponseDelayedinMins")
    .where(F.col("Neighborhood").isNotNull())
    .groupBy("Neighborhood", "CallType")
    .agg(
        F.avg("ResponseDelayedinMins").alias("AvgResponseDelayedinMins"),
        F.count("CallType").alias("TotalIncidents")
    )
    .orderBy(F.col("AvgResponseDelayedinMins").desc())
)

### Which week in 2018 had the most fire calls?

In [0]:
display(
    fire_ts_df_2018
    .select(
        F.weekofyear("IncidentDate").alias("IncidentWeek"),
        "CallType"
    )
    .groupBy("IncidentWeek")
    .agg(F.count("CallType").alias("CountOfIncidents"))
    .orderBy(F.col("CountOfIncidents").desc(), F.col("IncidentWeek"))
)

### Is there a relationship between neighborhood, zip code, and number of fire calls?

In [0]:
output_df = (
    fire_ts_df_2018
    .select("IncidentDate", "CallType", "Neighborhood", "ResponseDelayedinMins", "Zipcode")
    .where(F.col("Neighborhood").isNotNull() & F.col("Zipcode").isNotNull())
    .groupBy("Zipcode", "Neighborhood")
    .agg(
        F.avg("ResponseDelayedinMins").alias("AvgResponseDelayedinMins"),
        F.count("CallType").alias("TotalIncidents")
    )
    .orderBy(F.col("AvgResponseDelayedinMins").desc())
)

In [0]:
display(output_df)

### Optional next step
How could we store this transformed data as Parquet or as a table and read it back later?

### Persisting Data

In real data pipelines, transformed data is typically saved for reuse.

Two common options:
- Save as Parquet files (efficient, columnar format)
- Save as a managed table for querying with SQL

In [0]:
output_path = "/Volumes/jenny-demo/my_schema/external_data_source_files/fire_calls_output_parquet"

output_df.write.mode("overwrite").parquet(output_path)     # .write.parquet(path) → saves optimized file format

In [0]:
df_parquet = spark.read.parquet(output_path)

display(df_parquet)

**Table version**

In [0]:
output_df.write.mode("overwrite").saveAsTable("fire_calls_summary")       # .saveAsTable() → creates queryable table

display(spark.table("fire_calls_summary"))